### OpenSSL API Detection
This notebook detects OpenSSL usage in certificates through OpenSSL API names (functions, headers, library files) found in their texts, limited to certificates whose sec-certs keywords do not declare OpenSSL.

In [3]:
from sec_certs.dataset.cc import CCDataset
from sec_certs.dataset.fips import FIPSDataset



In [4]:
from pathlib import Path

print("Loading datasets from JSON")
cc_dataset = CCDataset.from_json("./datasets/CC_dataset.json")
cc_dataset.root_dir = Path("./datasets")
print(f"Loaded {len(cc_dataset)} CC certificates from JSON")

fips_dataset = FIPSDataset.from_json("./datasets/FIPS_dataset.json")
fips_dataset.root_dir = Path("./datasets")
print(f"Loaded {len(fips_dataset)} FIPS 140 certificates from JSON")


Loading datasets from JSON


Dataset was created with sec-certs version 0.4.1.post1.dev86+g770a181a4.d20260805 (older than your version 0.5.0.post1.dev4+g9ee2a63ba). To install the matching version: pip install sec-certs==0.4.1.post1.dev86+g770a181a4.d20260805


Loaded 6828 CC certificates from JSON


Dataset was created with sec-certs version 0.4.1.post1.dev138+ga5d8c122a.d20260911 (older than your version 0.5.0.post1.dev4+g9ee2a63ba). To install the matching version: pip install sec-certs==0.4.1.post1.dev138+ga5d8c122a.d20260911


Loaded 5542 FIPS 140 certificates from JSON


### 1. Load and Filter OpenSSL Functions
We load the list of OpenSSL functions extracted from the docs and filter for comprehensive API prefixes covering core crypto, algorithms, engine, utilities, and OpenSSL 3.x APIs.

In [5]:
import json
import re

with open("src/openssl_man3.json", "r") as f:
    all_functions = json.load(f)

# Comprehensive OpenSSL prefixes covering core crypto, algorithms, utilities, and 3.x APIs
prefixes = (
    "SSL_", "EVP_", "BIO_", "BN_", "X509_", "PEM_", "CMS_", "CRYPTO_", "ASN1_", "d2i_", "i2d_",
    "OSSL_", "OPENSSL_", "RSA_", "DSA_", "DH_", "EC_", "ECDSA_", "ECDH_", "ENGINE_",
    "ERR_", "RAND_", "HMAC_", "PKCS12_", "PKCS7_", "PKCS8_", "PKCS5_", "OCSP_", "UI_", "SRP_"
)

filtered_functions = [f for f in all_functions if f.startswith(prefixes)]
print(f"Total functions extracted: {len(all_functions)}")
print(f"Filtered high-confidence functions: {len(filtered_functions)}")


Total functions extracted: 5671
Filtered high-confidence functions: 5116


### 2. Pre-process OpenSSL Function Token Patterns and Artifact Indicators
To scan large certificate documents efficiently, we pre-process the function list into an $O(1)$ lookup set and compile regex patterns for function prefixes as well as OpenSSL architectural artifacts (shared libraries, header paths, and core structs). This avoids iterating over thousands of individual function strings and prevents false-positive substring collisions.

In [6]:
import re
from collections import Counter

# Pre-process functions into an O(1) lookup set
openssl_func_set = set(filtered_functions)

# Pre-compile a regex pattern matching words that start with any of the OpenSSL prefixes
# Word boundaries (\b) ensure complete token matching without substring collisions
prefix_pattern_str = r"\b(?:" + "|".join(map(re.escape, prefixes)) + r")[A-Za-z0-9_]+\b"
openssl_token_pattern = re.compile(prefix_pattern_str)

# Pre-compile pattern for OpenSSL structural artifacts: headers, libraries, and core types
artifact_pattern_str = r"\b(?:openssl/[a-zA-Z0-9_\.]+|libcrypto(?:\.so|\.a)?|libssl(?:\.so|\.a)?|ssleay32(?:\.dll)?|libeay32(?:\.dll)?|SSL_CTX|EVP_CIPHER_CTX|EVP_MD_CTX|BIO_METHOD|OPENSSL_CONF)\b"
openssl_artifact_pattern = re.compile(artifact_pattern_str, re.IGNORECASE)

print(f"Pre-compiled token pattern for {len(prefixes)} prefixes.")
print(f"Loaded {len(openssl_func_set)} unique functions into lookup set.")
print("Pre-compiled artifact pattern for OpenSSL libraries, headers, and types.")


Pre-compiled token pattern for 30 prefixes.
Loaded 5116 unique functions into lookup set.
Pre-compiled artifact pattern for OpenSSL libraries, headers, and types.


### 3. Define Extraction Logic
We define functions that check if a certificate explicitly declares OpenSSL via sec-certs keywords. If it does not, we scan its associated text documents (reports, security targets, and certificate summaries) for OpenSSL functions and artifacts.

In [7]:
import os
import pandas as pd
from pathlib import Path

def mentions_openssl_cc(cert):
    # Check if OpenSSL is in the CC report or ST keywords
    if cert.pdf_data.report_keywords and "crypto_library" in cert.pdf_data.report_keywords:
        if "OpenSSL" in cert.pdf_data.report_keywords["crypto_library"]:
            return True
    if cert.pdf_data.st_keywords and "crypto_library" in cert.pdf_data.st_keywords:
        if "OpenSSL" in cert.pdf_data.st_keywords["crypto_library"]:
            return True
    return False

def mentions_openssl_fips(cert):
    # Check if OpenSSL is in the FIPS policy keywords
    if cert.pdf_data.keywords and "crypto_library" in cert.pdf_data.keywords:
        if "OpenSSL" in cert.pdf_data.keywords["crypto_library"]:
            return True
    return False

def declared_crypto_libraries(*keyword_dicts):
    # Collect library names declared in the crypto_library keywords of the given documents
    libraries = set()
    for kw in keyword_dicts:
        if kw and "crypto_library" in kw:
            libraries.update(kw["crypto_library"].keys())
    return sorted(libraries)

def safe_get_txt_path(target):
    """Safely extract file path from DocumentState, Path, or str without raising ValueError."""
    if target is None:
        return None
    if hasattr(target, "_txt_path") and target._txt_path is not None:
        return target._txt_path
    if hasattr(target, "txt_path"):
        try:
            return target.txt_path
        except (AttributeError, ValueError):
            return None
    return target

def scan_text_for_functions(target, functions=openssl_func_set, token_pattern=openssl_token_pattern, artifact_pattern=openssl_artifact_pattern):
    text_path = safe_get_txt_path(target)
    if not text_path or not os.path.exists(text_path):
        return {}
    
    with open(text_path, "r", encoding="utf-8", errors="ignore") as f:
        content = f.read()
        
    func_set = functions if isinstance(functions, set) else set(functions)
    matches = {}
    
    # Fast token extraction + set lookup using pre-compiled pattern
    if token_pattern is not None:
        tokens = token_pattern.findall(content)
        matches.update(Counter(t for t in tokens if t in func_set))
    elif isinstance(functions, (set, list)):
        for func in functions:
            count = content.count(func)
            if count > 0:
                matches[func] = count
                
    # Match architectural and binary artifacts
    if artifact_pattern is not None:
        matches.update(Counter(artifact_pattern.findall(content)))
        
    return dict(matches)


### 4. Test Extraction Logic on a Subset (Quick Debugging)
To test and debug quickly without scanning all certificates, configure `SAMPLE_SIZE` to scan a smaller slice of the datasets.

In [8]:
SAMPLE_SIZE = 100  # Set to an integer (e.g., 50 or 100) for fast testing

cc_sample = list(cc_dataset)[:SAMPLE_SIZE]
fips_sample = list(fips_dataset)[:SAMPLE_SIZE]

print("Testing logic on subset...")

sample_results = []
for cert in cc_sample:
    if not mentions_openssl_cc(cert):
        r_matches = scan_text_for_functions(cert.state.report)
        st_matches = scan_text_for_functions(cert.state.st)
        c_matches = scan_text_for_functions(getattr(cert.state, "cert", None))
        matches = {**r_matches}
        for k, v in st_matches.items():
            matches[k] = matches.get(k, 0) + v
        for k, v in c_matches.items():
            matches[k] = matches.get(k, 0) + v
        if matches:
            sample_results.append({
                "Dataset": "CC",
                "ID": cert.dgst,
                "Name": cert.name,
                "Manufacturer": cert.manufacturer,
                "MatchedFunctions": list(matches.keys()),
                "FunctionCount": len(matches),
                "TotalOccurrences": sum(matches.values())
            })

for cert in fips_sample:
    if not mentions_openssl_fips(cert):
        p_matches = scan_text_for_functions(getattr(cert.state, "policy", None))
        if p_matches:
            sample_results.append({
                "Dataset": "FIPS",
                "ID": cert.dgst,
                "Name": cert.web_data.module_name if hasattr(cert, "web_data") and cert.web_data else None,
                "Manufacturer": cert.web_data.vendor if hasattr(cert, "web_data") and cert.web_data else None,
                "MatchedFunctions": list(p_matches.keys()),
                "FunctionCount": len(p_matches),
                "TotalOccurrences": sum(p_matches.values())
            })

print(f"Subset scan found {len(sample_results)} matches:")
import pandas as pd
display(pd.DataFrame(sample_results))


Testing logic on subset...
Subset scan found 2 matches:


,Dataset,ID,Name,Manufacturer,MatchedFunctions,FunctionCount,TotalOccurrences
0,CC,5f4852891f6c0a97,PSTgateways Framework version 4.16.8-A,"Autek Ingeniería, S.L.","[libssl.so, libcrypto.so]",2,4
1,CC,157dfb97aa089aa7,YouWipe Data Erasure Tool 5.0 and WipeCenter C...,AllWipe Oy Ltd.,[RSA_sign],1,6


### 5. Scan Full Common Criteria Dataset


In [9]:
results = []

print("Scanning CC Dataset...")
for cert in cc_dataset:
    if not mentions_openssl_cc(cert):
        # Check report, target, and certificate text
        report_matches = scan_text_for_functions(cert.state.report)
        st_matches = scan_text_for_functions(cert.state.st)
        cert_matches = scan_text_for_functions(getattr(cert.state, "cert", None))
        
        # Combine matches
        all_matches = {**report_matches}
        for k, v in st_matches.items():
            all_matches[k] = all_matches.get(k, 0) + v
        for k, v in cert_matches.items():
            all_matches[k] = all_matches.get(k, 0) + v
            
        if all_matches:
            results.append({
                "Dataset": "CC",
                "ID": cert.dgst,
                "Manufacturer": cert.manufacturer,
                "Name": cert.name,
                "IssueDate": cert.not_valid_before,
                "CryptoLibraries": declared_crypto_libraries(cert.pdf_data.report_keywords, cert.pdf_data.st_keywords),
                "MatchedFunctions": list(all_matches.keys()),
                "FunctionCount": len(all_matches),
                "TotalOccurrences": sum(all_matches.values())
            })
print(f"Found {len(results)} CC certificates mentioning OpenSSL functions / artifacts.")


Scanning CC Dataset...
Found 108 CC certificates mentioning OpenSSL functions / artifacts.


### 6. Scan Full FIPS 140 Dataset


In [10]:
print("Scanning FIPS Dataset...")
fips_match_count = 0
for cert in fips_dataset:
    if not mentions_openssl_fips(cert):
        policy_matches = scan_text_for_functions(getattr(cert.state, "policy", None))
        if policy_matches:
            results.append({
                "Dataset": "FIPS",
                "ID": cert.dgst,
                "Manufacturer": cert.web_data.vendor if hasattr(cert, "web_data") and cert.web_data else None,
                "Name": cert.web_data.module_name if hasattr(cert, "web_data") and cert.web_data else None,
                "IssueDate": cert.web_data.validation_history[0].date if hasattr(cert, "web_data") and cert.web_data and cert.web_data.validation_history else None,
                "CryptoLibraries": declared_crypto_libraries(cert.pdf_data.keywords),
                "MatchedFunctions": list(policy_matches.keys()),
                "FunctionCount": len(policy_matches),
                "TotalOccurrences": sum(policy_matches.values())
            })
            fips_match_count += 1
print(f"Found {fips_match_count} FIPS certificates mentioning OpenSSL functions/artifacts.")
print(f"Found {len(results)} total certificates across CC and FIPS.")


Scanning FIPS Dataset...
Found 142 FIPS certificates mentioning OpenSSL functions/artifacts.
Found 250 total certificates across CC and FIPS.


### 7. Classify Matches
OpenSSL API names alone do not prove OpenSSL is used: forks and API-compatible libraries (BoringSSL, AWS-LC, LibreSSL, RSA BSAFE, wolfSSL) share the same names. Each matched certificate is assigned one class, checked in this order:
1. **OpenSSL derivative**: a derivative library is declared in the keywords or named in the text, or the text includes `openssl/service_indicator.h`, which exists only in BoringSSL and AWS-LC.
2. **OpenSSL named, not in keywords**: the text names OpenSSL (prose, CLI, file or header names), but sec-certs keyword extraction did not record it.
3. **API names only**: OpenSSL API names appear, but no library is named.

In [11]:
cc_by_id = {cert.dgst: cert for cert in cc_dataset}
fips_by_id = {cert.dgst: cert for cert in fips_dataset}

openssl_name_pattern = re.compile(r"\bopenssl\b", re.IGNORECASE)
derived_library_pattern = re.compile(r"\b(BoringSSL|AWS-LC|LibreSSL|BSAFE|wolfSSL)\b", re.IGNORECASE)
derived_library_names = {"boringssl": "BoringSSL", "aws-lc": "AWS-LC", "libressl": "LibreSSL", "bsafe": "BSAFE", "wolfssl": "wolfSSL"}
derived_keyword_libraries = {"BoringSSL", "LibreSSL", "wolfSSL"}

def read_cert_texts(entry):
    # Concatenate all text documents that were scanned for this entry
    if entry["Dataset"] == "CC":
        cert = cc_by_id[entry["ID"]]
        documents = [cert.state.report, cert.state.st, getattr(cert.state, "cert", None)]
    else:
        cert = fips_by_id[entry["ID"]]
        documents = [getattr(cert.state, "policy", None)]
    content = ""
    for document in documents:
        text_path = safe_get_txt_path(document)
        if text_path and os.path.exists(text_path):
            with open(text_path, "r", encoding="utf-8", errors="ignore") as f:
                content += f.read()
    return content

for entry in results:
    content = read_cert_texts(entry)
    derived = {derived_library_names[m.lower()] for m in derived_library_pattern.findall(content)}
    derived |= {lib for lib in entry["CryptoLibraries"] if lib in derived_keyword_libraries}
    if "openssl/service_indicator.h" in content:
        derived.add("BoringSSL/AWS-LC (service_indicator.h)")
    entry["OpenSSLTextMentions"] = len(openssl_name_pattern.findall(content))
    entry["DerivedLibraries"] = sorted(derived)
    if derived:
        entry["Classification"] = "OpenSSL derivative"
    elif entry["OpenSSLTextMentions"]:
        entry["Classification"] = "OpenSSL named, not in keywords"
    else:
        entry["Classification"] = "API names only"

display(pd.crosstab(pd.Series([e["Classification"] for e in results], name="Classification"),
                    pd.Series([e["Dataset"] for e in results], name="Dataset"), margins=True))

Dataset,CC,FIPS,All
Classification,,,
API names only,54,68,122
OpenSSL derivative,45,67,112
"OpenSSL named, not in keywords",9,7,16
All,108,142,250


### 8. Display Results and Export Matched Entries


In [12]:
import pandas as pd
from pathlib import Path

df_matches = pd.DataFrame(results)

if not df_matches.empty:
    df_matches = df_matches.sort_values(by="TotalOccurrences", ascending=False)
    display(df_matches.head(25))
    print(f"Found {len(df_matches)} certificates using OpenSSL API names without OpenSSL in their keywords.")
    
    output_dir = Path("./results")
    output_dir.mkdir(parents=True, exist_ok=True)
    
    csv_path = output_dir / "openssl_api_detection_matched_entries.csv"
    json_path = output_dir / "openssl_api_detection_matched_entries.json"
    
    df_matches.to_csv(csv_path, index=False)
    df_matches.to_json(json_path, orient="records", indent=4)
    print(f"Exported matched entries to {csv_path} and {json_path}")
else:
    print("No undeclared OpenSSL API usage found.")


,Dataset,ID,Manufacturer,Name,IssueDate,CryptoLibraries,MatchedFunctions,FunctionCount,TotalOccurrences,OpenSSLTextMentions,DerivedLibraries,Classification
215,FIPS,cdad86ca9450d1c1,CoCo Communications,The CoCo Crypto Module,2008-09-02,[],"[DSA_generate_key, DH_generate_key, DSA_genera...",251,259,0,[],API names only
166,FIPS,b3b130b344c28e52,Brocade Communications Systems LLC,Brocade Fabric OS FIPS Cryptographic Module 8.2,2018-07-25,[],"[RAND_status, DH_OpenSSL, DH_get_default_metho...",42,83,0,[],API names only
126,FIPS,51283fa355970ea4,Stryker Corporation,Stryker Badge Cryptographic Module,2026-06-17,[],"[OPENSSL_cleanse, OSSL_PROVIDER_self_test, OSS...",7,72,0,[],API names only
223,FIPS,23c657f2efda7731,Tumbleweed Communications Corp.,ValiCert Security Module,2003-02-04,[],"[PEM_read_bio_X509, PEM_read_bio_X509_CRL, PEM...",33,72,0,[],API names only
218,FIPS,a07e64a2cea4932c,Dell EMC,EMC Data Domain Crypto-C Micro Edition,2016-09-26,[],"[BIO_append_filename, BIO_clear_flags, BIO_cle...",59,59,0,[],API names only
177,FIPS,fc5101a7f55d71e2,RSA,RSA BSAFE® Crypto-C Micro Edition,2014-12-16,[],"[BIO_append_filename, BIO_clear_flags, BIO_cle...",59,59,0,[BSAFE],OpenSSL derivative
181,FIPS,642c2a0918053df9,"RSA, The Security Division of EMC",RSA BSAFE(R) Crypto-C Micro Edition,2014-03-05,[],"[BIO_append_filename, BIO_clear_flags, BIO_cle...",59,59,0,[BSAFE],OpenSSL derivative
162,FIPS,967ae5f964cddff3,RSA,RSA BSAFE® Crypto-C Micro Edition,2014-12-12,[],"[BIO_get_buffer_num_lines, BIO_append_filename...",59,59,0,[BSAFE],OpenSSL derivative
229,FIPS,88100ba34db736c9,"RSA, The Security Division of EMC",RSA BSAFE(R) Crypto-C Micro Edition,2013-12-05,[],"[BIO_append_filename, BIO_clear_flags, BIO_cle...",59,59,0,[BSAFE],OpenSSL derivative
191,FIPS,adde7bf0e33b1136,"RSA, The Security Division of EMC",RSA BSAFE(R) Crypto-C Micro Edition,2013-11-25,[],"[BIO_append_filename, BIO_clear_flags, BIO_cle...",59,59,0,[BSAFE],OpenSSL derivative


Found 250 certificates using OpenSSL API names without OpenSSL in their keywords.
Exported matched entries to results/openssl_api_detection_matched_entries.csv and results/openssl_api_detection_matched_entries.json
